# Urban Heat Island Mapping with MODIS Land Surface Temperature

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/leafmap/blob/master/docs/notebooks/117_urban_heat_island.ipynb)

This notebook demonstrates how to visualize and analyze **Urban Heat Island (UHI)** effects using MODIS MOD11A2 Land Surface Temperature (LST) data and [leafmap](https://leafmap.org).

The Urban Heat Island effect describes how urban areas exhibit significantly higher temperatures than surrounding rural land due to human activity, impervious surfaces, and reduced vegetation. LST derived from thermal infrared satellite sensors is one of the most widely used proxies for quantifying UHI intensity.

**Outline:**
- Install and import dependencies
- Download MODIS MOD11A2 LST data via `earthaccess`
- Load and preprocess the LST GeoTIFF
- Apply scale factor and mask fill values
- Visualize LST on an interactive leafmap
- Compute a split map comparing urban vs. rural thermal patterns
- Export results

## Install dependencies

Uncomment if running in Google Colab or a fresh environment.

In [ ]:
# %pip install leafmap earthaccess rioxarray matplotlib numpy

## Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr
import earthaccess
import leafmap

## Authenticate with NASA Earthdata

A free NASA Earthdata account is required. Register at [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov). Credentials are stored in `~/.netrc` after first login.

In [ ]:
earthaccess.login(persist=True)

## Search for MODIS MOD11A2 LST data

MOD11A2 provides 8-day composite Land Surface Temperature at 1 km resolution. We search for a summer granule over Lahore, Pakistan — one of South Asia's most prominent Urban Heat Island cities.

In [ ]:
# Lahore, Pakistan bounding box [west, south, east, north]
bbox = (73.8, 31.3, 74.6, 31.7)

results = earthaccess.search_data(
    short_name="MOD11A2",
    version="061",
    bounding_box=bbox,
    temporal=("2023-06-01", "2023-06-30"),
)

print(f"Found {len(results)} granule(s)")
results[0]

## Download the LST granule

In [ ]:
files = earthaccess.download(results[:1], local_path="modis_lst")
hdf_path = files[0]
print(f"Downloaded: {hdf_path}")

## Load and preprocess LST

MOD11A2 stores LST as scaled integers. The LST Day layer (`LST_Day_1km`) must be multiplied by **0.02** to convert to Kelvin, then subtract 273.15 for Celsius. Fill value **0** is masked out.

In [ ]:
# Open the daytime LST subdataset
lst_raw = rxr.open_rasterio(
    f'HDF4_EOS:EOS_GRID:"{hdf_path}":MODIS_Grid_8Day_1km_LST:LST_Day_1km',
    masked=True,
).squeeze()

# Apply scale factor and convert to Celsius
lst_kelvin = lst_raw.where(lst_raw != 0) * 0.02
lst_celsius = lst_kelvin - 273.15

print(
    f"LST range: {float(lst_celsius.min()):.1f} °C  to  {float(lst_celsius.max()):.1f} °C"
)
lst_celsius

## Quick static plot

A preliminary view before loading into leafmap.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
lst_celsius.plot(
    ax=ax,
    cmap="RdYlBu_r",
    robust=True,
    cbar_kwargs={"label": "Land Surface Temperature (°C)"},
)
ax.set_title("MODIS MOD11A2 — Daytime LST, Lahore, June 2023", fontsize=13)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("lahore_lst.png", dpi=150)
plt.show()

## Export preprocessed LST as GeoTIFF

In [ ]:
lst_output = "lahore_lst_celsius.tif"
lst_celsius.rio.to_raster(lst_output)
print(f"Saved: {lst_output}")

## Visualize on an interactive leafmap

Load the LST raster with a diverging colormap. Warmer colours (red/orange) indicate urban heat pockets; cooler blues indicate vegetated or rural land.

In [ ]:
m = leafmap.Map(center=[31.5, 74.2], zoom=10)
m.add_basemap("OpenStreetMap")

m.add_raster(
    lst_output,
    colormap="RdYlBu_r",
    layer_name="LST Day (°C)",
    opacity=0.75,
)

m.add_colorbar(
    cmap="RdYlBu_r",
    label="Land Surface Temperature (°C)",
    position="bottomright",
)

m

## Split map — LST vs. Satellite basemap

The split map widget lets you swipe between the LST layer and the satellite basemap, making it easy to correlate high-temperature zones with built-up surfaces.

In [ ]:
m2 = leafmap.Map(center=[31.5, 74.2], zoom=10)

left_layer = leafmap.basemap_to_tiles(leafmap.basemaps.SATELLITE)
right_layer = leafmap.basemap_to_tiles(leafmap.basemaps.OpenStreetMap)

m2.split_map(
    left_layer=left_layer,
    right_layer=right_layer,
)

m2.add_raster(
    lst_output,
    colormap="RdYlBu_r",
    layer_name="LST Day (°C)",
    opacity=0.7,
)

m2

## Compute UHI intensity

A simple UHI index: difference between each pixel and the scene mean LST. Positive values = warmer than average (urban heat zones).

In [ ]:
scene_mean = float(lst_celsius.mean())
uhi_index = lst_celsius - scene_mean

uhi_output = "lahore_uhi_index.tif"
uhi_index.rio.to_raster(uhi_output)

print(f"Scene mean LST: {scene_mean:.2f} °C")
print(f"Max UHI intensity: +{float(uhi_index.max()):.2f} °C")
print(f"Min UHI intensity: {float(uhi_index.min()):.2f} °C")

## Visualize UHI index

In [ ]:
m3 = leafmap.Map(center=[31.5, 74.2], zoom=10)
m3.add_basemap("SATELLITE")

m3.add_raster(
    uhi_output,
    colormap="seismic",
    layer_name="UHI Index (°C above mean)",
    opacity=0.75,
)

m3.add_colorbar(
    cmap="seismic",
    label="UHI Index (°C above scene mean)",
    position="bottomright",
)

m3

## Summary

This notebook demonstrated how to:

1. Search and download MODIS MOD11A2 LST data using `earthaccess`
2. Apply scale factors and unit conversions with `rioxarray`
3. Visualize LST interactively with `leafmap` and a diverging colormap
4. Build a split map to compare LST against basemap layers
5. Compute a simple Urban Heat Island intensity index

The workflow is applicable to any city worldwide — change `bbox` and `temporal` to explore different regions and seasons.

## References

- [MODIS MOD11A2 Product Page](https://lpdaac.usgs.gov/products/mod11a2v061/) — LP DAAC
- [earthaccess](https://earthaccess.readthedocs.io/) — NASA Earthdata Python library
- [leafmap](https://leafmap.org/) — Wu, 2021
- [rioxarray](https://corteva.github.io/rioxarray/)